
GOLD LAYER — DAILY SALES AGGREGATION
Table: gold.daily_sales

 Grain:
 1 row = 1 day

 Business goal:
 Track daily revenue, order volume, and sales trends.
 Enable time-based analysis:
 - Monthly trends
 - Seasonality
 - Weekday vs weekend performance


1. Load Source Data

In [0]:

# Fact table (transaction-level data)
df_fact = spark.table("gold.fact_sales")

2. Daily Aggregation

In [0]:
from pyspark.sql.functions import sum, count

df_daily_sales = df_fact.groupBy("order_date").agg(
    sum("revenue").alias("daily_revenue"),        # total revenue per day
    sum("quantity").alias("daily_quantity"),      # total items sold per day
    count("order_id").alias("total_orders")       # number of orders per day
)

3. Enrich with Date Dimension

In [0]:

# Load date dimension
df_date = spark.table("gold.dim_date")

# Join to add time attributes (year, month, weekday, etc.)
df_daily_sales = df_daily_sales.join(
    df_date,
    df_daily_sales["order_date"] == df_date["date"],
    "left"
)
df_daily_sales = df_daily_sales.drop("date")

4. Data Quality Checks

In [0]:

null_dates = df_daily_sales.filter("order_date IS NULL").count()
negative_revenue = df_daily_sales.filter("daily_revenue < 0").count()

print(f"Null order_date: {null_dates}")
print(f"Negative revenue rows: {negative_revenue}")

5. Write to Gold Table

In [0]:

spark.sql("CREATE DATABASE IF NOT EXISTS gold")


spark.sql("DROP TABLE IF EXISTS gold.daily_sales")

df_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.daily_sales")

6. Validation

In [0]:


display(spark.table("gold.daily_sales"))

In [0]:
display(
    spark.table("gold.daily_sales")
    .orderBy("order_date")
)